In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

import pingouin as pg

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon

### Data Loading

DATA_DIR = Path("/Users/nolan/Documents/Data/UID")

Computed Metrics

In [6]:
METRICS = [
    "surp_mean", "surp_slor", "uid_std", "uid_mad", "uid_pwd",
    "uid_range", "uid_slope", "uid_len", "surp_peak", "uid_iqr",
    "uid_skew", "uid_gini", "uid_roughness", "uid_cv",
]

In [9]:
def load_and_pair(path):
    df = pd.read_csv(path, index_col=0, low_memory=False)
    df[["fact", "doc_name", "conv_id", "conv_type"]] = df["doc_id"].str.split("::", expand=True)

    merge_cols = [c for c in ["doc_name", "sent_idx", "uid_level", "context"] if c in df.columns]
    metrics    = [m for m in METRICS if m in df.columns]

    factual = df[df["fact"] == "f"]
    cf      = df[df["fact"] == "cf"]

    paired = cf.merge(factual[merge_cols + metrics], on=merge_cols, suffixes=("_cf", "_f"))
    for m in metrics:
        paired[f"delta_{m}"] = paired[f"{m}_cf"] - paired[f"{m}_f"]

    paired["genre"] = paired["doc_name"].str.split("_").str[1]
    print(f"  {path.name}: {len(cf):,} CF rows → {len(paired):,} paired ({len(paired)/len(cf):.0%})")
    return paired

In [11]:
right = load_and_pair(DATA_DIR / "sweep_right_word.csv")
nxt   = load_and_pair(DATA_DIR / "next_sent_uid.csv")
# ctx   = load_and_pair(DATA_DIR / "sweep_context_word.csv")

  sweep_right_word.csv: 9,655 CF rows → 9,630 paired (100%)
  next_sent_uid.csv: 2,297 CF rows → 2,297 paired (100%)


In [14]:
base = right[right["uid_level"] == "(-0,+0)"]

print("Conversion type counts (right sweep, no added context):")
print(base["conv_type"].value_counts().to_string())
print()

Conversion type counts (right sweep, no added context):
conv_type
a>p    2300
p>a     139



In [16]:
rows = []
for label, df in [("s_t  (right sweep, -0+0)", base), ("s_t+1 (next sentence)", nxt)]:
    for ct in ["a>p", "p>a"]:
        d = df[df["conv_type"] == ct]["delta_surp_mean"].dropna()
        rows.append({"dataset": label, "conv_type": ct, "n": len(d),
                     "median_Δ": round(d.median(), 4), "mean_Δ": round(d.mean(), 4)})

pd.DataFrame(rows)

,dataset,conv_type,n,median_Δ,mean_Δ
0,"s_t (right sweep, -0+0)",a>p,2300,0.2043,0.3616
1,"s_t (right sweep, -0+0)",p>a,139,0.9301,1.0948
2,s_t+1 (next sentence),a>p,2164,0.0066,0.0625
3,s_t+1 (next sentence),p>a,133,0.0073,0.0429


### Graph Formatting

In [12]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120